In [27]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
import jieba
import numpy as np
from sklearn.impute import SimpleImputer

# 1 DictVectorizer示例：将字典列表转换为特征矩阵

In [ ]:
# DictVectorizer示例：将字典列表转换为特征矩阵

# 创建一个字典列表，每个字典代表一个样本
data = [
    {'城市': '北京', '温度': 100, '湿度': 60, '风力': '强'},
    {'城市': '上海', '温度': 90, '湿度': 70, '风力': '中'},
    {'城市': '深圳', '温度': 110, '湿度': 80, '风力': '弱'}
]

# 初始化DictVectorizer，设置sparse=False返回密集矩阵而非稀疏矩阵
dict_vec = DictVectorizer(sparse=False)
dict_vec #对象

,dtype,<class 'numpy.float64'>
,separator,'='
,sparse,False
,sort,True


In [ ]:

# fit_transform 接口通常用于将原始数据转换为适合机器学习模型处理的格式，例如在特征工程中，将分类数据转换为数值型的 one-hot 编码。
feature_matrix = dict_vec.fit_transform(data) # 转成one-hot编码型的ndarray

# 查看特征名称，轴1
feature_names = dict_vec.get_feature_names_out()

print("特征名称:", feature_names)
print("特征矩阵:\n", feature_matrix)
print("特征矩阵形状:", feature_matrix.shape)
print('特征类型',type(feature_matrix))

特征名称: ['城市=上海' '城市=北京' '城市=深圳' '温度' '湿度' '风力=中' '风力=弱' '风力=强']
特征矩阵:
 [[  0.   1.   0. 100.  60.   0.   0.   1.]
 [  1.   0.   0.  90.  70.   1.   0.   0.]
 [  0.   0.   1. 110.  80.   0.   1.   0.]]
特征矩阵形状: (3, 8)
特征类型 <class 'numpy.ndarray'>


In [ ]:

# `inverse_transform` 接口将特征矩阵转换回字典形式的功能可应用于多个场景，例如：转换器
# 1. 模型解释：当使用机器学习模型进行预测后，需要理解模型是基于哪些原始特征做出的决策。通过将特征矩阵转换回字典形式，可以清晰地看到每个样本对应的原始特征及其取值。
# 2. 数据可视化：在进行数据可视化时，通常需要原始的字典数据格式来更方便地展示数据。将特征矩阵转换回字典形式可以为可视化操作提供合适的数据结构。
# 3. 数据导出：如果需要将处理后的数据导出到其他系统或工具中，而这些系统或工具更适合处理字典形式的数据，那么就可以使用该接口进行转换。

original_data = dict_vec.inverse_transform(feature_matrix)
print("\n转换回的字典数据:")
for item in original_data:
    print(item)


转换回的字典数据:
{'城市=北京': 1.0, '温度': 100.0, '湿度': 60.0, '风力=强': 1.0}
{'城市=上海': 1.0, '温度': 90.0, '湿度': 70.0, '风力=中': 1.0}
{'城市=深圳': 1.0, '温度': 110.0, '湿度': 80.0, '风力=弱': 1.0}


# 2CountVectorizer英文示例：将英文文本转换为词频矩阵

In [8]:
# CountVectorizer英文示例：将英文文本转换为词频矩阵

# 准备一些简单的英文文本数据
english_texts = [
    "Machine learning is a branch of artificial intelligence",
    "Deep learning is a method of machine learning",
    "Natural language processing is an important application of artificial intelligence"
]

# 初始化CountVectorizer,空格，标点符号，都认为是分隔符，单个字母，认为没有语义
# CountVectorizer用于将文本集合转换为词频矩阵。
# 它会将文本中的每个单词作为一个特征，统计每个单词在每个文档中出现的次数。
# 这里我们初始化一个 CountVectorizer 的实例，使用默认的参数配置。
english_count_vec = CountVectorizer()
english_count_vec

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


In [11]:
# 转换文本数据为词频矩阵
english_X = english_count_vec.fit_transform(english_texts)

# 获取特征名称（词汇表），轴1所属列名
english_vocabulary = english_count_vec.get_feature_names_out()

# 将稀疏矩阵转换为密集矩阵以便于显示
english_X_dense = english_X.toarray()

print("英文词汇表:", english_vocabulary)
print("英文词频矩阵:\n", english_X_dense)
print("英文矩阵形状:", english_X_dense.shape)

英文词汇表: ['an' 'application' 'artificial' 'branch' 'deep' 'important'
 'intelligence' 'is' 'language' 'learning' 'machine' 'method' 'natural'
 'of' 'processing']
英文词频矩阵:
 [[0 0 1 1 0 0 1 1 0 1 1 0 0 1 0]
 [0 0 0 0 1 0 0 1 0 2 1 1 0 1 0]
 [1 1 1 0 0 1 1 1 1 0 0 0 1 1 1]]
英文矩阵形状: (3, 15)


In [12]:
# 分析结果
print("\n文档-词条矩阵解释:")
for i, doc in enumerate(english_texts):
    print(f"文档 {i+1}: {doc}")
    print("包含的词条:")
    for j, term in enumerate(english_vocabulary):
        if english_X_dense[i, j] > 0:
            print(f"  - '{term}' 出现 {english_X_dense[i, j]} 次")
    print()



文档-词条矩阵解释:
文档 1: Machine learning is a branch of artificial intelligence
包含的词条:
  - 'artificial' 出现 1 次
  - 'branch' 出现 1 次
  - 'intelligence' 出现 1 次
  - 'is' 出现 1 次
  - 'learning' 出现 1 次
  - 'machine' 出现 1 次
  - 'of' 出现 1 次

文档 2: Deep learning is a method of machine learning
包含的词条:
  - 'deep' 出现 1 次
  - 'is' 出现 1 次
  - 'learning' 出现 2 次
  - 'machine' 出现 1 次
  - 'method' 出现 1 次
  - 'of' 出现 1 次

文档 3: Natural language processing is an important application of artificial intelligence
包含的词条:
  - 'an' 出现 1 次
  - 'application' 出现 1 次
  - 'artificial' 出现 1 次
  - 'important' 出现 1 次
  - 'intelligence' 出现 1 次
  - 'is' 出现 1 次
  - 'language' 出现 1 次
  - 'natural' 出现 1 次
  - 'of' 出现 1 次
  - 'processing' 出现 1 次



In [ ]:
# jieba 分词的基于词典的最大匹配算法与数据结构中的 KMP 匹配存在明显差别。
# 在目的上，jieba 分词的最大匹配算法主要用于将一段文本按照词典中的词语进行合理切分，以得到有意义的词语序列，辅助自然语言处理中的后续任务，如词性标注、语义分析等。
# 而 KMP 匹配主要用于在一个长文本（主串）中查找某个特定子串是否存在及其位置。
# 在匹配方式上，基于词典的最大匹配算法会根据词典中词语的长度，从文本的某一端（正向或逆向）开始，尽可能匹配最长的词语。例如正向最大匹配是从文本左侧开始，尝试匹配最长的词。
# 而 KMP 匹配是通过构建一个部分匹配表（PMT）来避免在匹配过程中进行不必要的回溯，利用已经匹配过的部分信息，提高匹配效率，它是逐字符进行比较的。


# jieba 分词主要有以下几种分词算法及原理：
# 1. 基于词典的最大匹配算法
# 最大匹配算法是一种基于词典的分词方法，它将待分词的文本与词典中的词语进行匹配。
# 最大匹配算法又分为正向最大匹配（FMM）、逆向最大匹配（BMM）和双向最大匹配（BiMM）。
# 正向最大匹配是从文本的左侧开始，尽可能匹配词典中最长的词语；
# 逆向最大匹配则是从文本的右侧开始；
# 双向最大匹配会同时进行正向和逆向最大匹配，然后根据一定的规则选择最优的分词结果。

# 2. 基于隐马尔可夫模型（HMM）的分词算法
# HMM 是一种统计模型，在 jieba 中用于处理未登录词（词典中不存在的词）的分词。
# 它将分词问题看作是一个序列标注问题，每个字在分词中有四种可能的状态：B（词首）、M（词中）、E（词尾）、S（单字成词）。
# 通过训练语料统计出状态转移概率和发射概率，然后使用 Viterbi 算法找出最优的状态序列，从而完成分词。

# 3. 基于条件随机场（CRF）的分词算法（jieba 未原生实现，但可结合使用）
# CRF 是一种判别式概率无向图模型，同样用于序列标注任务。
# 与 HMM 不同，CRF 可以利用上下文的全局特征，而不仅仅是局部特征。
# 它通过学习特征函数和权重，计算整个序列的概率，从而得到最优的分词结果。


In [ ]:
# jieba分词示例

# 准备一些中文文本
text = "我爱北京天安门，天安门上太阳升。"

# 默认分词
print("默认分词:") #jieba分词内部有很多分词算法
seg_list = jieba.cut(text, cut_all=False) #返回的是迭代器,字符串
print("默认模式: " + " ".join(seg_list))


默认分词:
默认模式: 我 爱 北京 天安门 ， 天安门 上 太阳升 。


In [ ]:

# # 全模式分词
# print("\n全模式分词:")
# seg_list = jieba.cut(text, cut_all=True)
# print("全模式: " + "/ ".join(seg_list))

# # 搜索引擎模式
# print("\n搜索引擎模式:")
# seg_list = jieba.cut_for_search(text)
# print("搜索引擎模式: " + "/ ".join(seg_list))

# # 添加自定义词典
# print("\n添加自定义词典:")
# jieba.add_word("天安门上")
# seg_list = jieba.cut(text, cut_all=False)
# print("添加自定义词典后: " + "/ ".join(seg_list))

# # 词性标注
# print("\n词性标注:")
# import jieba.posseg as pseg
# words = pseg.cut(text)
# print("词性标注结果:")
# for word, flag in words:
#     print(f"{word} ({flag})")


In [ ]:
# CountVectorizer示例：将文本转换为词频矩阵

# 准备一些简单的中文文本数据
texts = [
    "机器学习是人工智能的一个分支",
    "深度学习是机器学习的一种方法",
    "自然语言处理是人工智能的重要应用"
]

# 初始化CountVectorizer
count_vec = CountVectorizer()

# 转换文本数据为词频矩阵
X = count_vec.fit_transform(texts)
X #稀疏矩阵


<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3 stored elements and shape (3, 3)>

In [14]:

# 获取特征名称（词汇表）
vocabulary = count_vec.get_feature_names_out()

# 将稀疏矩阵转换为密集矩阵以便于显示
X_dense = X.toarray()

print("词汇表:", vocabulary)
print("词频矩阵:\n", X_dense)
print("矩阵形状:", X_dense.shape)

词汇表: ['机器学习是人工智能的一个分支' '深度学习是机器学习的一种方法' '自然语言处理是人工智能的重要应用']
词频矩阵:
 [[1 0 0]
 [0 1 0]
 [0 0 1]]
矩阵形状: (3, 3)


In [ ]:
# 使用jieba分词器处理中文
print("\n使用jieba分词的CountVectorizer:")
# 创建一个CountVectorizer对象，使用jieba分词器对输入文本进行分词处理，并且设置最小文档频率为2，即只考虑在至少2个文档中出现的词项
count_vec_jieba = CountVectorizer(analyzer=lambda x: jieba.cut(x),min_df=2)

# 使用创建好的CountVectorizer对象对文本数据进行拟合和转换，将文本数据转换为词频矩阵ndarray
X_jieba = count_vec_jieba.fit_transform(texts)
# 从拟合好的CountVectorizer对象中获取词汇表，即所有被考虑的词项列表
vocabulary_jieba = count_vec_jieba.get_feature_names_out()

print("jieba分词词汇表:", vocabulary_jieba)
print("jieba分词词频矩阵:\n", X_jieba.toarray())
print("jieba分词矩阵形状:", X_jieba.toarray().shape)


使用jieba分词的CountVectorizer:
jieba分词词汇表: ['人工智能' '学习' '是' '机器' '的']
jieba分词词频矩阵:
 [[1 1 1 1 1]
 [0 2 1 1 1]
 [1 0 1 0 1]]
jieba分词矩阵形状: (3, 5)


# 3 tfidf

In [ ]:
# TF-IDF处理示例：将文本转换为TF-IDF特征矩阵

# 使用jieba分词器进行TF-IDF处理
print("\n使用jieba分词的TF-IDF处理:")
# TfidfVectorizer 是 sklearn 库中用于将文本数据转换为 TF-IDF 特征矩阵的类。
# TF-IDF（Term Frequency-Inverse Document Frequency）是一种常用的文本特征提取方法
# 用于评估一个词在文档集合中的重要性。log对数化处理，返回的是小数类型的ndarray


# analyzer 参数指定了文本分析器，这里使用了一个匿名函数 lambda x: jieba.cut(x)，
# 意味着使用 jieba 分词器对输入的文本 x 进行分词处理。
# 最终将这个配置好的 TfidfVectorizer 实例赋值给变量 tfidf_vec_jieba。

tfidf_vec_jieba = TfidfVectorizer(analyzer=lambda x: jieba.cut(x))

X_tfidf_jieba = tfidf_vec_jieba.fit_transform(texts) # 文本列表转换为TF-IDF矩阵

tfidf_vocabulary_jieba = tfidf_vec_jieba.get_feature_names_out()

print("jieba分词词汇表:", tfidf_vocabulary_jieba)
# print('TF-IDF特征矩阵没有toarray',X_tfidf_jieba)
print("jieba分词TF-IDF特征矩阵:\n", X_tfidf_jieba.toarray())
print("jieba分词矩阵形状:", X_tfidf_jieba.toarray().shape)


Building prefix dict from the default dictionary ...



使用jieba分词的TF-IDF处理:


Dumping model to file cache C:\Users\hsl\AppData\Local\Temp\jieba.cache
Loading model cost 0.870 seconds.
Prefix dict has been built successfully.


jieba分词词汇表: ['一个' '一种' '人工智能' '分支' '处理' '学习' '应用' '方法' '是' '机器' '深度' '的' '自然语言' '重要']
jieba分词TF-IDF特征矩阵:
 [[0.47496141 0.         0.3612204  0.47496141 0.         0.3612204
  0.         0.         0.28051986 0.3612204  0.         0.28051986
  0.         0.        ]
 [0.         0.38955498 0.         0.         0.         0.5925332
  0.         0.38955498 0.23007745 0.2962666  0.38955498 0.23007745
  0.         0.        ]
 [0.         0.         0.3311001  0.         0.43535684 0.
  0.43535684 0.         0.25712876 0.         0.         0.25712876
  0.43535684 0.43535684]]
jieba分词矩阵形状: (3, 14)


# 归一化

In [ ]:
# 导入归一化相关的库
# 归一化虽然有很多优点，但也存在一些缺点：
# 1. 对异常值敏感：归一化方法通常基于数据的最大值和最小值，当数据中存在异常值时，这些异常值会对归一化的结果产生较大影响，导致归一化后的数据分布发生扭曲。
# 2. 不适合有离群点的数据：如果数据集中存在离群点，使用归一化可能会使大部分正常数据集中在一个很小的区间内，从而丢失数据的部分特征信息。
# 3. 缺乏鲁棒性：在数据更新或者有新的数据加入时，由于最大值和最小值可能会发生变化，需要重新进行归一化操作，这增加了计算成本和复杂度。

import numpy as np

# 创建一个简单的数据矩阵作为示例
print("归一化示例:")
X = np.array([
    [1, -1, 2],
    [2, 0, 0],
    [0, 1, -1],
    [5, 2, 0]
])
print("原始数据矩阵:\n", X)
print("数据矩阵形状:", X.shape)

# MinMaxScaler归一化：将数据缩放到[0,1]区间
min_max_scaler = MinMaxScaler()
X_minmax = min_max_scaler.fit_transform(X)
print("\nMinMaxScaler归一化后的数据(缩放到[0,1]):\n", X_minmax)


归一化示例:
原始数据矩阵:
 [[ 1 -1  2]
 [ 2  0  0]
 [ 0  1 -1]
 [ 5  2  0]]
数据矩阵形状: (4, 3)

MinMaxScaler归一化后的数据(缩放到[0,1]):
 [[0.2        0.         1.        ]
 [0.4        0.33333333 0.33333333]
 [0.         0.66666667 0.        ]
 [1.         1.         0.33333333]]


In [ ]:
# 也可以缩放到其他区间，例如[-1,1]
min_max_scaler_custom = MinMaxScaler(feature_range=(-1, 1))
X_minmax_custom = min_max_scaler_custom.fit_transform(X)
print("MinMaxScaler归一化后的数据(缩放到[-1,1]):\n", X_minmax_custom)

MinMaxScaler归一化后的数据(缩放到[-1,1]):
 [[-0.6        -1.          1.        ]
 [-0.2        -0.33333333 -0.33333333]
 [-1.          0.33333333 -1.        ]
 [ 1.          1.         -0.33333333]]


# 标准化

In [19]:
# StandardScaler标准化：将数据转换为均值为0，标准差为1的分布
# 标准化（StandardScaler标准化）的优点：
# 1. 消除特征之间的量纲影响，使得不同特征具有可比性。例如在多元线性回归中，不同特征的单位可能不同，标准化后能避免因量纲差异导致某些特征对模型的影响过大。
# 2. 加速模型收敛。在梯度下降等迭代算法中，标准化后的数据能使算法更快地找到最优解，因为数据分布更加集中，梯度下降的步长可以更合理地选择。
# 3. 提高模型的稳定性。对于一些对数据尺度敏感的模型，如支持向量机（SVM），标准化能使模型在不同数据集上的表现更加稳定。

# 标准化（StandardScaler标准化）的缺点：
# 1. 改变了原始数据的分布。标准化后的数据不再保留原始数据的分布特征，对于一些需要考虑数据原始分布的分析场景可能不适用。
##### 2. 对异常值敏感。由于标准化是基于数据的均值和标准差进行计算的，异常值会对均值和标准差产生较大影响，从而影响标准化的结果。
# 3. 增加计算复杂度。在处理大规模数据集时，计算均值和标准差需要遍历整个数据集，会增加计算时间和内存开销。



print("\nStandardScaler标准化示例:")
X_example = np.array([[1., -1., 3.],
                      [2., 4., 2.],
                      [4., 6., -1.]])
print("原始数据矩阵:\n", X_example)

# 应用StandardScaler进行标准化
std_scaler = StandardScaler()
X_std = std_scaler.fit_transform(X_example)
print("StandardScaler标准化后的数据:\n", X_std)


StandardScaler标准化示例:
原始数据矩阵:
 [[ 1. -1.  3.]
 [ 2.  4.  2.]
 [ 4.  6. -1.]]
StandardScaler标准化后的数据:
 [[-1.06904497 -1.35873244  0.98058068]
 [-0.26726124  0.33968311  0.39223227]
 [ 1.33630621  1.01904933 -1.37281295]]


In [20]:

# 查看标准化后的均值和标准差
print("标准化后的均值:", X_std.mean(axis=0)) #无限接近0
print("标准化后的标准差:", X_std.std(axis=0))

# 也可以查看原始数据的均值和标准差
print("\n原始数据的均值:", X_example.mean(axis=0))
print("原始数据的标准差:", X_example.std(axis=0))


标准化后的均值: [-1.48029737e-16  7.40148683e-17  7.40148683e-17]
标准化后的标准差: [1. 1. 1.]

原始数据的均值: [2.33333333 3.         1.33333333]
原始数据的标准差: [1.24721913 2.94392029 1.69967317]


In [23]:
# SimpleImputer是scikit-learn库中的一个类，用于处理数据集中的缺失值。
# 它提供了多种填充策略，如均值（'mean'）、中位数（'median'）、常数（'constant'）和最频繁值（'most_frequent'）等。
# 使用时，需要先创建SimpleImputer对象并指定填充策略，然后调用fit_transform方法对包含缺失值的数据进行填充。
print("\nSimpleImputer示例:")
# 创建一个包含缺失值的数据矩阵
X_missing = np.array([
    [np.nan, 2, 3],
    [4, np.nan, 6],
    [7, 8, np.nan],
    [np.nan, np.nan, 6]
])

# 使用均值策略填充缺失值
mean_imputer = SimpleImputer(missing_values=np.nan,strategy='mean')
X_imputed_mean = mean_imputer.fit_transform(X_missing)
print("\n使用均值填充后的数据矩阵:\n", X_imputed_mean)



SimpleImputer示例:

使用均值填充后的数据矩阵:
 [[5.5 2.  3. ]
 [4.  5.  6. ]
 [7.  8.  5. ]
 [5.5 5.  6. ]]


In [ ]:

# 使用中位数策略填充缺失值
median_imputer = SimpleImputer(strategy='median')
X_imputed_median = median_imputer.fit_transform(X_missing)
print("\n使用中位数填充后的数据矩阵:\n", X_imputed_median)


使用中位数填充后的数据矩阵:
 [[5.5 2.  3. ]
 [4.  5.  6. ]
 [7.  8.  6. ]
 [5.5 5.  6. ]]


In [ ]:

# 使用常数值填充缺失值
constant_imputer = SimpleImputer(strategy='constant', fill_value=0) # 常数值填充缺失值，需要fill_value参数
X_imputed_constant = constant_imputer.fit_transform(X_missing)
print("\n使用常数0填充后的数据矩阵:\n", X_imputed_constant)


使用常数0填充后的数据矩阵:
 [[0. 2. 3.]
 [4. 0. 6.]
 [7. 8. 0.]
 [0. 0. 6.]]


In [26]:

# 使用最频繁值填充缺失值,众数
most_frequent_imputer = SimpleImputer(strategy='most_frequent')
X_imputed_most_frequent = most_frequent_imputer.fit_transform(X_missing)
print("\n使用最频繁值填充后的数据矩阵:\n", X_imputed_most_frequent)



使用最频繁值填充后的数据矩阵:
 [[4. 2. 3.]
 [4. 2. 6.]
 [7. 8. 6.]
 [4. 2. 6.]]


# 方差阈值

In [28]:
# `VarianceThreshold` 是 `sklearn.feature_selection` 模块中的一个接口，主要用于特征选择任务。
# 其核心功能是通过设定一个方差阈值，将数据集中方差低于该阈值的特征去除。
# 方差反映了特征的离散程度，方差较小的特征往往包含较少的有用信息，去除这些特征可以减少数据维度，提高模型训练效率和性能。
print("\nVarianceThreshold示例:")

# 创建一个示例数据矩阵
X_variance = np.array([
    [0, 2, 0, 3],
    [0, 1, 4, 3],
    [0, 1, 1, 3]
])

# 计算每个特征的方差
feature_variances = np.var(X_variance, axis=0)
print("\n各特征的方差:", feature_variances)



VarianceThreshold示例:

各特征的方差: [0.         0.22222222 2.88888889 0.        ]


In [29]:

# 使用方差阈值为0.8进行特征选择
# 这将移除方差小于0.8的特征
selector = VarianceThreshold(threshold=0.8)
X_selected = selector.fit_transform(X_variance)

# 显示保留的特征索引
print("\n保留的特征索引:", selector.get_support(indices=True))
print("保留的特征方差:", feature_variances[selector.get_support()])
print("\n特征选择后的数据矩阵:\n", X_selected)


保留的特征索引: [2]
保留的特征方差: [2.88888889]

特征选择后的数据矩阵:
 [[0]
 [4]
 [1]]


In [30]:

# 尝试不同的阈值
selector_low = VarianceThreshold(threshold=0.1)
X_selected_low = selector_low.fit_transform(X_variance)
print("\n阈值为0.1时保留的特征索引:", selector_low.get_support(indices=True))
print("阈值为0.1时特征选择后的数据矩阵:\n", X_selected_low)


阈值为0.1时保留的特征索引: [1 2]
阈值为0.1时特征选择后的数据矩阵:
 [[2 0]
 [1 4]
 [1 1]]


# PCA

In [31]:
# 函数接收输入数据矩阵和要保留的主成分数量或比例作为参数，
# 执行PCA拟合和转换操作，并输出原始数据、转换后的数据、方差信息以及主成分解释的方差比例等，
# 最后返回PCA模型、转换后的数据和解释方差比例。

# 定义一个理解PCA的示例函数，该函数用于通过示例演示主成分分析（PCA）的过程
def understand_pca(data, n_components=None):
    """
    通过示例理解PCA主成分分析
    
    参数:
    data: 输入数据矩阵
    n_components: 要保留的主成分比例，如果是0到1之间的浮点数，表示保留的主成分解释的方差比例。如果是整数，表示保留的列数量。
    
    返回:
    pca模型、转换后的数据、解释方差比例
    """
    import numpy as np
    import matplotlib.pyplot as plt
    
    # 创建PCA模型
    pca = PCA(n_components=n_components)
    
    # 拟合数据并转换
    transformed_data = pca.fit_transform(data)
    
    # 输出原始数据和结果
    print("原始数据:\n", data)
    print("\n数据形状:", data.shape)
    
    # 输出PCA的结果
    print("\nPCA转换后的数据:\n", transformed_data)
    print("\n转换后数据形状:", transformed_data.shape)
    #输出transformed_data的方差
    print("\n转换后数据方差:", np.var(transformed_data, axis=0))
    #输出transformed_data的方差和
    print("\n转换后数据方差和:", np.var(transformed_data, axis=0).sum())
    
    # 主成分解释的方差比例
    print("\n各主成分解释的方差比例:", pca.explained_variance_ratio_)
    print("累计解释的方差比例:", np.sum(pca.explained_variance_ratio_))
    
    return pca, transformed_data, pca.explained_variance_ratio_


In [34]:

# 使用上下文中的数据
X = np.array([[2, 8, 4, 5],
              [6, 3, 0, 8],
              [5, 4, 9, 1]])

#输出X的方差
print("\n原始数据方差:", np.var(X, axis=0))
#输出X的方差和
print("\n原始数据方差和:", np.var(X, axis=0).sum())


原始数据方差: [ 2.88888889  4.66666667 13.55555556  8.22222222]

原始数据方差和: 29.333333333333336


In [35]:

# 调用函数演示PCA
print("完整PCA示例:")
pca_full, data_full, var_ratio_full = understand_pca(X)

完整PCA示例:
原始数据:
 [[2 8 4 5]
 [6 3 0 8]
 [5 4 9 1]]

数据形状: (3, 4)

PCA转换后的数据:
 [[-1.28620952e-15  3.82970843e+00  5.26052119e-16]
 [-5.74456265e+00 -1.91485422e+00  5.26052119e-16]
 [ 5.74456265e+00 -1.91485422e+00  5.26052119e-16]]

转换后数据形状: (3, 3)

转换后数据方差: [2.20000000e+01 7.33333333e+00 6.48230758e-63]

转换后数据方差和: 29.333333333333332

各主成分解释的方差比例: [7.50000000e-01 2.50000000e-01 9.43400565e-33]
累计解释的方差比例: 1.0


In [36]:
# 降维到2个主成分
print("\n\n降维到2个主成分的PCA示例:")
pca_2d, data_2d, var_ratio_2d = understand_pca(X, n_components=0.9)



降维到2个主成分的PCA示例:
原始数据:
 [[2 8 4 5]
 [6 3 0 8]
 [5 4 9 1]]

数据形状: (3, 4)

PCA转换后的数据:
 [[-1.28620952e-15  3.82970843e+00]
 [-5.74456265e+00 -1.91485422e+00]
 [ 5.74456265e+00 -1.91485422e+00]]

转换后数据形状: (3, 2)

转换后数据方差: [22.          7.33333333]

转换后数据方差和: 29.333333333333332

各主成分解释的方差比例: [0.75 0.25]
累计解释的方差比例: 1.0
